# BB-Swin-Unet for Kvasir-SEG Polyp Segmentation (PyTorch)

Same two-stage pipeline as your BB-UNet Kvasir notebook -- YOLO extracts bounding boxes, then a second model consumes them through gated skip connections -- but with **BBSwinUnet** swapped in for the CNN-based BB-UNet, and every training setting (image size, loss, optimizer, scheduler, patience, metrics) matched to your uploaded `main.ipynb` Swin-Unet script rather than re-derived.

1. **YOLO stage** (Sections 1-6) -- identical approach to your BB-UNet Kvasir notebook: connected-component bounding boxes from the binary masks, `nc=1` (`polyp`), train YOLOv8n, then run it to build single-channel binary bbox maps.
2. **BB-Swin-Unet stage** (Sections 7-16) -- dataset with Albumentations (same fixes as the BB-UNet notebook: bbmap resized to each image's native size before the transform, and the stray trailing channel dim squeezed off), `BBSwinUnet` (gated skip connections via `BBConvGate2D`), `DiceBCELoss` on logits, and the same metric stack as `main.ipynb` (Jaccard, Dice/F1, Accuracy, Precision, Recall, Specificity, Hausdorff/HD95).

**Requirements**: `bb_swin_unet.py` and `swin_transformer_unet_skip_expand_decoder_sys.py` must be importable (same folder as this notebook, or on your `PYTHONPATH`).

**Settings matched to your `main.ipynb`** (not re-derived): `IMG_SIZE=224` (Swin-Unet's own constraint -- 224/patch4=56, which divides cleanly by window_size=7 across all 4 stages; your BB-UNet notebook used 256, which doesn't tile evenly for Swin), `batch_size=16`, `lr=1e-3`, `num_epochs=100`, early-stop `patience=40`, scheduler `ReduceLROnPlateau(patience=50)`, loss = `DiceBCELoss(dice_weight=0.5, bce_weight=0.5)` on raw logits (Swin-Unet's output has no built-in sigmoid, same as your vanilla script).

**What's carried over from the BB-UNet Kvasir notebook** (not from `main.ipynb`, since that one has no bbox stage at all): the entire YOLO pipeline (Sections 1-6), and the two dataset bugfixes discovered while debugging that notebook (native-size bbmap resize, trailing channel-dim squeeze) -- applied here proactively since the same Albumentations multi-mask-target setup is used.


## 1. Environment & Imports

In [ ]:
import os
import cv2
import yaml
import random
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics -- same stack as main.ipynb
from torchmetrics import JaccardIndex, F1Score, Accuracy, Precision, Recall, Specificity
from medpy.metric import binary as medpy_binary

# BB-Swin-Unet -- requires bb_swin_unet.py and swin_transformer_unet_skip_expand_decoder_sys.py
# to be importable (same folder as this notebook, or on your PYTHONPATH)
from bb_swin_unet import BBSwinUnet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


## 2. Configuration

In [ ]:
# TODO: set this to your actual Kvasir-SEG path
DATA_DIR = r"C:\\Users\\PC\\Desktop\\unetcodes\\Kvasir-SEG"
IMAGES_DIR = os.path.join(DATA_DIR, "images")
MASKS_DIR = os.path.join(DATA_DIR, "masks")

# Swin-Unet constraint (matches main.ipynb): with patch_size=4 and window_size=7,
# 224 -> 56 -> 28 -> 14 -> 7 across the 4 stages divides cleanly. 256 (your BB-UNet
# notebook's size) does NOT tile evenly for Swin's window attention -- don't reuse it here.
IMG_SIZE = 224
NUM_CLASSES = 1                 # binary output: logits, 1 channel (background vs polyp)
BB_CHANNELS = 1                 # single object class for YOLO ("polyp")
CLASS_NAMES = ["polyp"]         # YOLO class names

# Same 70/15/15 split as your BB-UNet notebook and main.ipynb
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Matched to main.ipynb
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
NUM_EPOCHS = 100
PATIENCE = 40            # early-stop patience (main.ipynb's value; the BB-UNet notebook used 50)
SCHEDULER_PATIENCE = 50   # ReduceLROnPlateau patience (main.ipynb's value)
RANDOM_SEED = 42

YOLO_ROOT = os.path.join(os.getcwd(), "yolo_dataset")
YOLO_IMG_TRAIN = os.path.join(YOLO_ROOT, "images", "train")
YOLO_IMG_VAL = os.path.join(YOLO_ROOT, "images", "val")
YOLO_LBL_TRAIN = os.path.join(YOLO_ROOT, "labels", "train")
YOLO_LBL_VAL = os.path.join(YOLO_ROOT, "labels", "val")
for d in [YOLO_IMG_TRAIN, YOLO_IMG_VAL, YOLO_LBL_TRAIN, YOLO_LBL_VAL]:
    os.makedirs(d, exist_ok=True)

os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("results", exist_ok=True)


## 3. Pair Images with Masks & Split (train/val/test)
Same split reused for both the YOLO stage and BB-Swin-Unet, so the test set is never seen during training of either model.

In [ ]:
image_files = sorted(os.listdir(IMAGES_DIR))
mask_files = sorted(os.listdir(MASKS_DIR))

image_mask_pairs = []
for image_file in image_files:
    image_path = os.path.join(IMAGES_DIR, image_file)
    mask_path = os.path.join(MASKS_DIR, image_file)   # Kvasir masks share the image's filename
    if os.path.isfile(mask_path):
        image_mask_pairs.append((image_path, mask_path))

print(f"Paired {len(image_mask_pairs)} image/mask files")
del image_files, mask_files

rng = random.Random(RANDOM_SEED)
pairs_shuffled = image_mask_pairs.copy()
rng.shuffle(pairs_shuffled)

n = len(pairs_shuffled)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * (TRAIN_RATIO + VAL_RATIO))

train_pairs = pairs_shuffled[:n_train]
val_pairs = pairs_shuffled[n_train:n_val]
test_pairs = pairs_shuffled[n_val:]

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")


## 4. Extract Bounding Boxes From Masks & Build the YOLO Dataset
Boxes are extracted per connected component (a mask can contain more than one polyp blob), all labeled class `0` (`polyp`). Only `train_pairs`/`val_pairs` go into the YOLO dataset -- `test_pairs` stays untouched by YOLO training too.

In [ ]:
def mask_to_yolo_boxes(mask_hw, img_w, img_h, min_area=16):
    """Returns a list of (class_id, x_center, y_center, width, height), normalized
    to [0,1], one box per connected component (all class 0 = 'polyp')."""
    binary = (mask_hw > 0).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)

    boxes = []
    for label_id in range(1, num_labels):  # skip background label 0
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        x = stats[label_id, cv2.CC_STAT_LEFT]
        y = stats[label_id, cv2.CC_STAT_TOP]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]

        xc = (x + w / 2.0) / img_w
        yc = (y + h / 2.0) / img_h
        wn = w / img_w
        hn = h / img_h
        boxes.append((0, xc, yc, wn, hn))
    return boxes


def write_yolo_label(txt_path, boxes):
    with open(txt_path, 'w') as f:
        for c, xc, yc, w, h in boxes:
            f.write(f"{c} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")


def build_yolo_split(pairs, img_out_dir, lbl_out_dir, split_name):
    n_written = 0
    for image_path, mask_path in pairs:
        basename = os.path.splitext(os.path.basename(image_path))[0]

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        img_h, img_w = mask.shape[:2]

        boxes = mask_to_yolo_boxes(mask, img_w, img_h)
        if not boxes:
            continue

        img = cv2.imread(image_path)
        cv2.imwrite(os.path.join(img_out_dir, basename + ".jpg"), img)
        write_yolo_label(os.path.join(lbl_out_dir, basename + ".txt"), boxes)
        n_written += 1

    print(f"{split_name}: wrote {n_written} image/label pairs")


build_yolo_split(train_pairs, YOLO_IMG_TRAIN, YOLO_LBL_TRAIN, "train")
build_yolo_split(val_pairs, YOLO_IMG_VAL, YOLO_LBL_VAL, "val")


In [ ]:
data_yaml = {
    "path": YOLO_ROOT,
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
yaml_path = os.path.join(YOLO_ROOT, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"Wrote {yaml_path}")
print(open(yaml_path).read())


## 5. Train YOLOv8n

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8n.pt")

yolo_results = yolo_model.train(
    data=yaml_path,
    epochs=100,
    imgsz=IMG_SIZE,
    patience=20,
    device=0,
    project="yolo_runs",
    name="polyp_detector",
    workers=0,   # IMPORTANT on Windows/Jupyter: num_workers>0 tries to re-spawn worker
                 # processes from the notebook's __main__ and silently deadlocks instead
                 # of erroring. 0 = single-process DataLoader, safe in a notebook kernel.
)

yolo_best_weights = os.path.join("yolo_runs", "polyp_detector", "weights", "best.pt")
print("Best YOLO weights:", yolo_best_weights)


## 6. Run Trained YOLO to Build Binary Bounding-Box Maps
Predicts on all paired images (train + val + test) so every split has a matching bbox map for BB-Swin-Unet.

In [ ]:
trained_yolo = YOLO(yolo_best_weights)

all_pairs = train_pairs + val_pairs + test_pairs
all_image_paths = [p for p, m in all_pairs]
yolo_predictions = trained_yolo.predict(source=all_image_paths, imgsz=IMG_SIZE, iou=0.7, conf=0.25, device=0)

box_list = []
for result in yolo_predictions:
    box_list.append(result.boxes.xywh.cpu().numpy())  # (x_center, y_center, w, h) in pixel coords at imgsz

bb_map_by_path = {}
for (image_path, _), boxes in zip(all_pairs, box_list):
    binary_map = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for x, y, w, h in boxes.tolist():
        x1, y1 = round(x - w / 2), round(y - h / 2)
        x2, y2 = round(x + w / 2), round(y + h / 2)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(IMG_SIZE, x2), min(IMG_SIZE, y2)
        binary_map[y1:y2, x1:x2] = 1
    binary_map = np.nan_to_num(binary_map, nan=0)
    bb_map_by_path[image_path] = binary_map

print(f"Built {len(bb_map_by_path)} bounding-box maps, shape {IMG_SIZE}x{IMG_SIZE}")
n_empty = sum(1 for m in bb_map_by_path.values() if m.sum() == 0)
print(f"Images with no detected box: {n_empty} / {len(bb_map_by_path)}")


## 7. Dataset Class (Albumentations, 3 synchronized targets: image / mask / bb map)
Same two fixes as the BB-UNet notebook: the bbmap (rasterized at YOLO's fixed `IMG_SIZE`) is resized to each image's native resolution before the synchronized transform, and any stray trailing channel dim Albumentations adds to `mask`/`bbmap` (from having two mask-type targets in one `Compose`) is squeezed off.

In [ ]:
class BBSwinUnetKvasirDataset(Dataset):
    def __init__(self, pairs, bb_map_by_path, transform=None):
        self.pairs = pairs
        self.bb_map_by_path = bb_map_by_path
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.uint8)

        # bbmap was rasterized at YOLO's fixed inference size (IMG_SIZE x IMG_SIZE),
        # but source images/masks vary in native resolution -- resize it (nearest,
        # since it's binary) to match this image's size before the synchronized transform.
        bbmap = self.bb_map_by_path[image_path]
        img_h, img_w = image.shape[:2]
        if bbmap.shape[:2] != (img_h, img_w):
            bbmap = cv2.resize(bbmap, (img_w, img_h), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            augmented = self.transform(image=image, mask=mask, bbmap=bbmap)
            image = augmented['image']          # (3, H, W) float tensor, normalized
            mask = augmented['mask']             # (H, W) tensor
            bbmap = augmented['bbmap']           # (H, W) tensor
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask = torch.from_numpy(mask).long()
            bbmap = torch.from_numpy(bbmap).long()

        # Albumentations can hand back a stray trailing channel dim (H,W,1) on the
        # primary 'mask' target when the Compose also has another mask-type
        # additional_target (here, 'bbmap') -- squeeze it off so mask is always (H,W).
        if mask.dim() == 3:
            mask = mask.squeeze(-1)
        if bbmap.dim() == 3 and bbmap.shape[-1] == 1:
            bbmap = bbmap.squeeze(-1)

        bbmap = bbmap.unsqueeze(0).float()       # (1, H, W)
        fused = torch.cat([bbmap, image], dim=0)  # (1+3, H, W) -- bb channel(s) first, then RGB
        return fused, mask.long()


train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'bbmap': 'mask'})

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'bbmap': 'mask'})

train_dataset = BBSwinUnetKvasirDataset(train_pairs, bb_map_by_path, transform=train_transform)
val_dataset = BBSwinUnetKvasirDataset(val_pairs, bb_map_by_path, transform=val_transform)
test_dataset = BBSwinUnetKvasirDataset(test_pairs, bb_map_by_path, transform=val_transform)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


## 8. Sanity Check: Visualize One Sample

In [ ]:
def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = img_np * std + mean
    return np.clip(img_np, 0, 1)


fused, mask = train_dataset[0]
bbmap_sample = fused[0].numpy()
image_sample = denormalize(fused[1:4])
mask_sample = mask.numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(image_sample); axes[0].set_title('Image'); axes[0].axis('off')
axes[1].imshow(mask_sample, cmap='gray'); axes[1].set_title('Ground-truth mask'); axes[1].axis('off')
axes[2].imshow(bbmap_sample, cmap='gray'); axes[2].set_title('YOLO-derived bbox map'); axes[2].axis('off')
plt.tight_layout()
plt.show()


## 9. DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


## 10. Model, Optimizer, Scheduler
Same `depths`/`num_heads`/`window_size`/`drop_path_rate` as `main.ipynb`'s `SwinUnet` call -- only the class changes, to `BBSwinUnet`, plus `bb_channels=1` for the bbox input.

In [ ]:
model = BBSwinUnet(
    img_size=IMG_SIZE,
    in_chans=3,
    num_classes=NUM_CLASSES,   # binary segmentation, logits (no built-in sigmoid)
    embed_dim=96,
    depths=[2, 2, 2, 2],
    num_heads=[3, 6, 12, 24],
    window_size=7,
    drop_path_rate=0.1,
    bb_channels=BB_CHANNELS,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=SCHEDULER_PATIENCE)

print(next(model.parameters()).device)


## 11. Loss Function: Dice + BCE (matches main.ipynb)
Operates on raw logits -- `BBSwinUnet.forward` doesn't apply a sigmoid internally, same as your vanilla `SwinUnet`.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super(DiceBCELoss, self).__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets):
        # inputs: (N,1,H,W) raw logits; targets: (N,H,W) long (0/1)
        inputs = inputs.squeeze(1)
        targets = targets.float()

        smooth = 1e-5
        probs = torch.sigmoid(inputs)
        intersection = (probs * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (probs.sum() + targets.sum() + smooth)

        bce_loss = self.bce(inputs, targets)

        return self.dice_weight * dice_loss + self.bce_weight * bce_loss


class FocalLoss(nn.Module):
    def __init__(self, gamma=2, alpha=None):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        inputs = inputs.squeeze(1)
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce)
        focal_loss = ((1 - pt) ** self.gamma * bce).mean()
        return focal_loss


# Choose which loss to use (main.ipynb uses DiceBCELoss)
criterion = DiceBCELoss(dice_weight=0.5, bce_weight=0.5).to(device)
# criterion = FocalLoss(gamma=2).to(device)


## 12. Metrics Setup (identical to main.ipynb)

In [ ]:
jaccard = JaccardIndex(task='binary').to(device)
dice = F1Score(task='binary').to(device)      # F1Score == Dice for binary
accuracy = Accuracy(task='binary').to(device)
precision = Precision(task='binary').to(device)
recall = Recall(task='binary').to(device)
specificity = Specificity(task='binary').to(device)


def compute_hausdorff(pred_mask, true_mask, spacing=(1, 1)):
    pred_binary = pred_mask.astype(np.uint8)
    true_binary = true_mask.astype(np.uint8)
    if np.sum(pred_binary) == 0 or np.sum(true_binary) == 0:
        return np.nan, np.nan
    hd = medpy_binary.hd(pred_binary, true_binary, voxelspacing=spacing)
    hd95 = medpy_binary.hd95(pred_binary, true_binary, voxelspacing=spacing)
    return hd, hd95


## 13. Training and Validation Functions

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total_pixels = 0

    for fused, masks in tqdm(dataloader, desc='Training'):
        fused = fused.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(fused)          # (N,1,H,W) raw logits
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Select the channel explicitly rather than relying on squeeze(1), which
        # silently no-ops if the channel dim isn't exactly 1 (e.g. after stale
        # kernel state) -- see the BB-UNet notebook's debugging notes for why.
        outputs_c0 = outputs[:, 0, :, :]
        assert outputs_c0.shape == masks.shape, (
            f"Shape mismatch: outputs {outputs_c0.shape} vs masks {masks.shape}. "
            "Restart the kernel and re-run all cells top-to-bottom."
        )
        preds = (torch.sigmoid(outputs_c0) > 0.5).long()
        correct += (preds == masks).sum().item()
        total_pixels += masks.numel()

    avg_loss = total_loss / len(dataloader)
    avg_acc = correct / total_pixels
    return avg_loss, avg_acc


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    jaccard.reset(); dice.reset(); accuracy.reset()
    precision.reset(); recall.reset(); specificity.reset()

    hd_list, hd95_list = [], []

    with torch.no_grad():
        for fused, masks in tqdm(dataloader, desc='Validation'):
            fused = fused.to(device)
            masks = masks.to(device)

            outputs = model(fused)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

            outputs_c0 = outputs[:, 0, :, :]
            assert outputs_c0.shape == masks.shape, (
                f"Shape mismatch: outputs {outputs_c0.shape} vs masks {masks.shape}."
            )
            preds = (torch.sigmoid(outputs_c0) > 0.5).long()

            jaccard.update(preds, masks)
            dice.update(preds, masks)
            accuracy.update(preds, masks)
            precision.update(preds, masks)
            recall.update(preds, masks)
            specificity.update(preds, masks)

            for i in range(fused.size(0)):
                pred_np = preds[i].cpu().numpy()
                mask_np = masks[i].cpu().numpy()
                hd, hd95 = compute_hausdorff(pred_np, mask_np)
                hd_list.append(hd)
                hd95_list.append(hd95)

    avg_loss = total_loss / len(dataloader)
    metrics = {
        'loss': avg_loss,
        'val_accuracy': accuracy.compute().cpu().item(),
        'jaccard': jaccard.compute().cpu().item(),
        'dice': dice.compute().cpu().item(),
        'precision': precision.compute().cpu().item(),
        'recall': recall.compute().cpu().item(),
        'specificity': specificity.compute().cpu().item(),
        'hd_mean': np.nanmean(hd_list),
        'hd95_mean': np.nanmean(hd95_list),
    }
    return metrics


## 14. Training Loop with CSV Logging

In [ ]:
log_file = "logs/bbswinunet_training_log.csv"
csv_columns = ['epoch', 'train_loss', 'train_accuracy', 'val_loss', 'val_accuracy',
               'jaccard', 'dice', 'precision', 'recall', 'specificity',
               'hd_mean', 'hd95_mean']
if not os.path.exists(log_file):
    pd.DataFrame(columns=csv_columns).to_csv(log_file, index=False)

best_val_loss = float('inf')
patience_counter = 0
history = defaultdict(list)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = validate_one_epoch(model, val_loader, criterion, device)
    scheduler.step(val_metrics['loss'])

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_metrics['val_accuracy']:.4f}")
    print(f"Jaccard: {val_metrics['jaccard']:.4f} | Dice: {val_metrics['dice']:.4f}")
    print(f"Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f} | "
          f"Specificity: {val_metrics['specificity']:.4f}")
    print(f"HD: {val_metrics['hd_mean']:.2f} | HD95: {val_metrics['hd95_mean']:.2f}")

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_acc)
    for k, v in val_metrics.items():
        history[k].append(v)

    row = {'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_acc, **val_metrics}
    pd.DataFrame([row]).to_csv(log_file, mode='a', header=False, index=False)

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        torch.save(model.state_dict(), 'models/best_bbswinunet.pth')
        print("Best BB-Swin-Unet model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

torch.save(model.state_dict(), 'models/final_bbswinunet.pth')


## 15. Plot Training Curves

In [ ]:
def plot_individual_curves(history):
    epochs = history['epoch']

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['loss'], label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curves')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig('results/bbswinunet_loss_curve.png', dpi=150); plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history['train_accuracy'], label='Train Accuracy')
    plt.plot(epochs, history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy Curves')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig('results/bbswinunet_accuracy_curve.png', dpi=150); plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history['jaccard'], label='Jaccard (IoU)')
    plt.plot(epochs, history['dice'], label='Dice')
    plt.xlabel('Epoch'); plt.ylabel('Score'); plt.title('Jaccard and Dice')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig('results/bbswinunet_jaccard_dice_curve.png', dpi=150); plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history['precision'], label='Precision')
    plt.plot(epochs, history['recall'], label='Recall')
    plt.plot(epochs, history['specificity'], label='Specificity')
    plt.xlabel('Epoch'); plt.ylabel('Score'); plt.title('Precision, Recall, Specificity')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig('results/bbswinunet_precision_recall_specificity.png', dpi=150); plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(epochs, history['hd_mean'], label='HD')
    plt.plot(epochs, history['hd95_mean'], label='HD95')
    plt.xlabel('Epoch'); plt.ylabel('Distance'); plt.title('Hausdorff Distance')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig('results/bbswinunet_hausdorff_curve.png', dpi=150); plt.show()

plot_individual_curves(history)


## 16. Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load('models/best_bbswinunet.pth'))
model.eval()

jaccard.reset(); dice.reset(); accuracy.reset()
precision.reset(); recall.reset(); specificity.reset()
hd_list, hd95_list = [], []
test_loss_total = 0

with torch.no_grad():
    for fused, masks in tqdm(test_loader, desc='Testing'):
        fused = fused.to(device)
        masks = masks.to(device)

        outputs = model(fused)
        loss = criterion(outputs, masks)
        test_loss_total += loss.item()

        outputs_c0 = outputs[:, 0, :, :]
        assert outputs_c0.shape == masks.shape, (
            f"Shape mismatch: outputs {outputs_c0.shape} vs masks {masks.shape}."
        )
        preds = (torch.sigmoid(outputs_c0) > 0.5).long()

        jaccard.update(preds, masks)
        dice.update(preds, masks)
        accuracy.update(preds, masks)
        precision.update(preds, masks)
        recall.update(preds, masks)
        specificity.update(preds, masks)

        for i in range(fused.size(0)):
            pred_np = preds[i].cpu().numpy()
            mask_np = masks[i].cpu().numpy()
            hd, hd95 = compute_hausdorff(pred_np, mask_np)
            hd_list.append(hd)
            hd95_list.append(hd95)

avg_test_loss = test_loss_total / len(test_loader)

jaccard_score = jaccard.compute().cpu().item()
dice_score = dice.compute().cpu().item()
acc_score = accuracy.compute().cpu().item()
prec_score = precision.compute().cpu().item()
rec_score = recall.compute().cpu().item()
spec_score = specificity.compute().cpu().item()
hd_mean = np.nanmean(hd_list)
hd95_mean = np.nanmean(hd95_list)

print("\n===== BB-Swin-Unet Test Set Results =====")
print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Accuracy: {acc_score:.4f}")
print(f"Jaccard: {jaccard_score:.4f}")
print(f"Dice: {dice_score:.4f}")
print(f"Precision: {prec_score:.4f}")
print(f"Recall: {rec_score:.4f}")
print(f"Specificity: {spec_score:.4f}")
print(f"HD: {hd_mean:.2f}")
print(f"HD95: {hd95_mean:.2f}")

test_results = pd.DataFrame([{
    'loss': avg_test_loss,
    'accuracy': acc_score,
    'jaccard': jaccard_score,
    'dice': dice_score,
    'precision': prec_score,
    'recall': rec_score,
    'specificity': spec_score,
    'hd': hd_mean,
    'hd95': hd95_mean,
}])
test_results.to_csv('results/bbswinunet_test_metrics.csv', index=False)
print("Test metrics saved to results/bbswinunet_test_metrics.csv")


## 17. Visualize Test Predictions

In [ ]:
def visualize_test_predictions(model, dataset, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    indices = random.sample(range(len(dataset)), num_samples)

    for row, idx in enumerate(indices):
        fused, mask = dataset[idx]
        fused_batch = fused.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(fused_batch)
            pred = (torch.sigmoid(output.squeeze()) > 0.5).cpu().numpy().astype(np.uint8)

        bbmap_np = fused[0].numpy()
        image_np = denormalize(fused[1:4])
        mask_np = mask.cpu().numpy()

        axes[row, 0].imshow(image_np); axes[row, 0].set_title('Image'); axes[row, 0].axis('off')
        axes[row, 1].imshow(bbmap_np, cmap='gray'); axes[row, 1].set_title('YOLO bbox map'); axes[row, 1].axis('off')
        axes[row, 2].imshow(mask_np, cmap='gray'); axes[row, 2].set_title('Ground Truth'); axes[row, 2].axis('off')
        axes[row, 3].imshow(pred, cmap='gray'); axes[row, 3].set_title('BB-Swin-Unet Prediction'); axes[row, 3].axis('off')

    plt.tight_layout()
    plt.savefig('results/bbswinunet_test_predictions.png', dpi=150)
    plt.show()

visualize_test_predictions(model, test_dataset, num_samples=3)


## 18. Save Final Model

In [ ]:
torch.save(model.state_dict(), 'models/bbswinunet_final.pth')
print("Saved to models/bbswinunet_final.pth")
